# Spec RAG — chunk labelling & retrieval tests

Verifies the temporary document RAG built by the ContractBuilder:
1. Sections are labelled with the correct `mns` (service) and `stage` (2 / 3).
2. The indexed chunks carry that metadata in Qdrant.
3. Retrieval works **without a filter**, **with a metadata filter**, and via
   **self-query (LLM-inferred filter)**.

Requires a running Qdrant on localhost:6333. Uses a temporary collection that
is dropped at the end.

In [ ]:
# Step 1 — Imports and setup
import os
from collections import Counter

from openapi_rulesbank.config import get_logger
from openapi_rulesbank.config.paths import TSPEC_DATA_TEST_FILE
from openapi_rulesbank.nodes.reader import reader_node
from openapi_rulesbank.tools.rag_tools import (
    _label_sections,
    _spec_factory,
    index_spec_sections,
    search_spec,
    drop_spec_collection,
    umbrella_mns_for,
)
from qdrant_3gpp_rag import retrieve

TARGET_SERVICE = 'ProvMnS'
STAGE3_CLAUSE  = '12.1'  # ProvMnS stage 3 block in the test document (28532)

logger = get_logger(__name__)
logger.info('Imports OK.')

In [ ]:
# Step 2 — Parse the document into sections (Reader)
reader_result = reader_node({
    'main_doc_path':       os.path.abspath(TSPEC_DATA_TEST_FILE),
    'auxiliary_doc_paths': [],
    'target_yaml':         TARGET_SERVICE,
})
sections = reader_result['parsed_sections']
print(f'{len(sections)} sections parsed.')

In [ ]:
# Step 3 — Inspect the labelling (mns / stage) BEFORE indexing
# _label_sections tags each section with its service (mns) and stage (2/3).
segments = _label_sections(sections, STAGE3_CLAUSE)

# Distribution of (mns, stage) across sections.
dist = Counter((s['mns'], s['stage']) for s in segments)
print('Sections per (mns, stage):')
for (mns, stage), n in sorted(dist.items(), key=lambda x: (-x[1])):
    print(f'  {n:>4}  stage={stage}  mns={mns!r}')

In [ ]:
# Step 4 — Spot-check specific sections
# The ProvMnS stage 2 operation getMOIAttributes lives under clause 11.1.x, and
# its stage 3 mapping under 12.1.x. Both must share the same mns; stages differ.
by_title = {s['title'].strip(): s for s in segments}

def show(prefix):
    for s in segments:
        if s['title'].strip().startswith(prefix):
            print(f"  {s['title'][:48]:48} | mns={s['mns'][:38]!r} stage={s['stage']}")
            return
    print(f'  (no section starting with {prefix!r})')

print('Stage 2 (definitions):')
show('11.1.1.2 getMOIAttributes')
show('11.1.1.2.2')
print('Stage 3 (solution set):')
show('12.1.1.1.3 Operation getMOIAttributes')
show('12.1.1.3.2.1.3.2 HTTP GET')

In [ ]:
# Step 5 — Index into a temporary collection and inspect the stored payload
collection = index_spec_sections(sections, STAGE3_CLAUSE)
print('collection:', collection)

client = _spec_factory().client
print('vectors:', client.count(collection).count)

# langchain_qdrant nests metadata under the 'metadata' payload key.
pts, _ = client.scroll(collection, limit=1, with_payload=True)
print('payload keys:', list(pts[0].payload.keys()))
print('metadata sample:', pts[0].payload.get('metadata'))

In [ ]:
# Step 6 — Retrieval WITHOUT a filter (plain dense search)
# Should always return chunks; may include other services (no scoping).
query = 'HTTP GET managed object attributes retrieve query parameters getMOIAttributes'
ctx_nofilter = search_spec(query, collection, k=4)
print('chars:', len(ctx_nofilter))
print('getMOIAttributes present?', 'getMOIAttributes' in ctx_nofilter)
print('--- first 400 chars ---')
print(ctx_nofilter[:400])

In [ ]:
# Step 7 — Retrieval WITH a metadata filter (mns + stage 2)
# This is what the Extractor uses: scope to the target service's stage 2 defs.
mns = umbrella_mns_for('12.1.1.3.2.1.3.2 HTTP GET', sections)
print('target mns:', repr(mns))

ctx_filtered = search_spec(query, collection, k=4, filters={'stage': '2', 'mns': mns})
print('docs returned?', bool(ctx_filtered))
print('getMOIAttributes present?', 'getMOIAttributes' in ctx_filtered)
print('--- first 500 chars ---')
print(ctx_filtered[:500])

assert ctx_filtered, 'Filtered retrieval returned nothing — metadata filter is broken.'

In [ ]:
# Step 8 — Retrieval via self-query (LLM infers the filter)
# Sanity check that the LLM-driven path also works against this collection.
# metadata_field_info describes the filterable fields for the LLM.
from openapi_rulesbank.config.llm_config import llm
from qdrant_3gpp_rag.self_query import AttributeInfo

field_info = [
    AttributeInfo(name='mns',   description="The Management Service a section belongs to (e.g. 'Generic provisioning management service')", type='string'),
    AttributeInfo(name='stage', description="The 3GPP stage of the section: '2' (abstract definitions) or '3' (HTTP solution set)", type='string'),
]

result = retrieve(
    'getMOIAttributes stage 2 definition for the provisioning management service',
    factory=_spec_factory(),
    collection_name=collection,
    llm=llm,
    k=4,
    metadata_field_info=field_info,
    document_content_description='3GPP management service specification sections',
)
print('generated query :', result.generated_query)
print('generated filter:', result.generated_filter)
print('docs returned   :', len(result.docs))
for d in result.docs[:2]:
    print('  ', d.page_content[:80].replace(chr(10), ' '))

In [ ]:
# Step 9 — Clean up the temporary collection
drop_spec_collection(collection)
print('dropped; still exists?', _spec_factory().client.collection_exists(collection))